# 小雅 Kaggle T4 · 单图音频数字人口播 MP4 实际验收

独立试验，与当前 DUIX 无关。本 Notebook 使用 **SadTalker** 将 GitHub 小雅肖像与已在 Kaggle 生成的真实 8.24s 语音结合，尝试产出带音轨的 MP4。最终生产版 DUIX 仍需单独验证。

需要 Settings 开启 GPU T4 与 Internet。GPU 额度由 Kaggle 决定；Notebook 失败不得伪报成功。SadTalker 代码来自 https://github.com/OpenTalker/SadTalker，首次会下载模型权重并可能遇到依赖兼容问题。

In [ ]:
import sys, os, pathlib, time, subprocess, json, hashlib, shutil, traceback
work=pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path.cwd()
out=work/"xiaoya_video_results";out.mkdir(parents=True,exist_ok=True)
t0=time.monotonic()
probe=subprocess.run(["nvidia-smi","--query-gpu=name,memory.used,memory.total","--format=csv,noheader,nounits"],capture_output=True,text=True)
print(probe.stdout or probe.stderr)
assert probe.returncode==0,"GPU not enabled"
print("Python",sys.version)


In [ ]:
%pip install -q "numpy<2" "scipy>=1.10,<1.13" "librosa>=0.10,<0.11" "numba" "imageio" "imageio-ffmpeg" "pydub" "resampy" "kornia==0.6.8" "face-alignment" "yacs" "scikit-image" "basicsr==1.4.2" "facexlib==0.3.0" "gfpgan" "av" "safetensors" "Pillow" "requests" "opencv-python-headless" "pyyaml" "tqdm" "soundfile"

In [ ]:
import requests
from PIL import Image
base="https://raw.githubusercontent.com/mjw3177109/xunigirl/main/"
def download(url,path,minimum=1024):
    path=pathlib.Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.stat().st_size >= minimum:return path
    r=requests.get(url,timeout=(25,180),stream=True);r.raise_for_status()
    with open(path,"wb") as f:
        for buf in r.iter_content(1024*1024):
            if buf:f.write(buf)
    assert path.stat().st_size>=minimum, f"Download too small: {url}"
    return path
image_file=download(base+"assets/portraits/xiaoya.png",work/"xiaoya.png",150000)
audio_file=download(base+"voices/xiaoya/ref_audio.wav",work/"xiaoya.wav",200000)
audio_sha=hashlib.sha256(audio_file.read_bytes()).hexdigest()
assert audio_sha=="5017add8a805d69b97a60d4a7caf16d0f087c8dbec0cf62e636154b1ee289feb", "Not the validated Kaggle-generated voice"
photo=Image.open(image_file).convert("RGB")
w,h=photo.size
# The existing full-body portrait has a small face; create a stable upper-body test crop.
# This is merely a preprocessed copy, never modifies the original source asset.
side=int(min(w,h)*0.67)
cx=w//2;top=int(h*0.05)
crop=photo.crop((max(0,cx-side//2),top,min(w,cx+side//2),min(h,top+side))).resize((640,640))
portrait=work/"xiaoya_closeup.png";crop.save(portrait)
print("input",photo.size,"closeup",crop.size,"audio_sha256",audio_sha)


In [ ]:
sadtalker=work/"SadTalker"
if not (sadtalker/"inference.py").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/OpenTalker/SadTalker.git",str(sadtalker)],check=True,timeout=180)
# Old SadTalker + new torchvision removed this path; patch only a known compatibility import.
import site
for top in [*site.getsitepackages(),site.getusersitepackages()]:
    p=pathlib.Path(top)/"basicsr/data/degradations.py"
    if p.is_file():
        src=p.read_text()
        before="from torchvision.transforms.functional_tensor import rgb_to_grayscale"
        after="from torchvision.transforms.functional import rgb_to_grayscale"
        if before in src:p.write_text(src.replace(before,after))
# NumPy 2 removed legacy aliases used by SadTalker. Patch the downloaded
# open-source project only; do not downgrade Kaggle's Python 3.13 runtime.
import re
patch_counts={}
for pyfile in (sadtalker/"src").rglob("*.py"):
    original=pyfile.read_text(encoding="utf-8")
    fixed=original.replace("np.VisibleDeprecationWarning",
        "np.exceptions.VisibleDeprecationWarning")
    for old,new in [("float","float64"),("int","int64"),("complex","complex128"),("bool","bool_"),("object","object_")]:
        fixed=re.sub(r"\bnp\."+old+r"\b","np."+new,fixed)
    fixed=fixed.replace("Image.ANTIALIAS","Image.Resampling.LANCZOS")
    if fixed!=original:
        pyfile.write_text(fixed,encoding="utf-8")
        patch_counts[str(pyfile.relative_to(sadtalker))]=1
assert patch_counts, "Legacy SadTalker source compatibility patch had no targets"
print("SadTalker compatibility patches:",sorted(patch_counts))
models=sadtalker/"checkpoints";models.mkdir(exist_ok=True)
release="https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/"
weights={
 "SadTalker_V0.0.2_256.safetensors":250000000,
 "mapping_00229-model.pth.tar":10000000,
}
for name,minimum in weights.items():
    path=download(release+name,models/name,minimum)
    print(name,path.stat().st_size)


In [ ]:
import torch
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"
start=time.monotonic()
args=[sys.executable,"inference.py",
 "--driven_audio",str(audio_file),
 "--source_image",str(portrait),
 "--result_dir",str(out/"render"),
 "--size","256","--batch_size","1",
 "--preprocess","crop","--still"]
env=os.environ.copy()
env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"
res=subprocess.run(args,cwd=str(sadtalker),env=env,text=True,capture_output=True,timeout=2400)
(out/"inference_stdout.txt").write_text(res.stdout or "",encoding="utf-8")
(out/"inference_stderr.txt").write_text(res.stderr or "",encoding="utf-8")
print("SadTalker exit",res.returncode,"elapsed_seconds",round(time.monotonic()-start,2))
print("stdout last:",(res.stdout or "")[-2000:])
print("stderr last:",(res.stderr or "")[-2600:])
if res.returncode!=0:raise RuntimeError("SadTalker inference failed. Inspect saved stderr/stdout in Kaggle Output.")
videos=sorted((out/"render").rglob("*.mp4"),key=lambda p:p.stat().st_size,reverse=True)
assert videos,"No MP4 output despite zero process exit"
shutil.copyfile(videos[0],out/"xiaoya_talking_pilot.mp4")
check=subprocess.run(["ffprobe","-v","error","-show_entries","format=duration:stream=codec_type,codec_name","-of","json",str(out/"xiaoya_talking_pilot.mp4")],capture_output=True,text=True,check=True)
meta=json.loads(check.stdout)
streams=[s["codec_type"] for s in meta.get("streams",[])]
duration=float(meta["format"]["duration"])
assert "video" in streams and "audio" in streams and 5<=duration<=15, f"Invalid MP4 streams/duration: {meta}"
report={"status":"actual_mp4_generated","engine":"SadTalker (NOT DUIX)","gpu":probe.stdout.strip(),
 "audio_sha256":audio_sha,"seconds":round(duration,3),"streams":meta.get("streams"),
 "elapsed_seconds":round(time.monotonic()-t0,2),"mp4_size_bytes":(out/"xiaoya_talking_pilot.mp4").stat().st_size}
(out/"video_report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(report)
